# 📋 Multi-Document Policy Assistant

## What We're Building

An assistant that searches across **multiple policy documents** (HR, IT, Travel)
and answers questions with **citations** showing which policy and section the
answer came from.

## Why This Matters

Real organizations have dozens of policy documents. Employees waste hours
searching through them. A policy assistant:
- Searches all policies at once
- Returns the **most relevant** sections (not just similar ones)
- Cites the exact source so answers can be verified

## Key Concept: Reranking

Basic RAG retrieves chunks by embedding similarity. But similarity ≠ relevance.

**Reranking** adds a second pass:
```
Query → Retrieve top-20 chunks (fast, approximate)
         → Rerank to top-5 (slow, precise)
           → Feed to LLM
```

The reranker scores each (query, chunk) pair more carefully than embeddings alone.
We'll implement a **LLM-based reranker** using Ollama — the LLM scores how relevant
each chunk is to the question.

## Stack
- **LangChain** — retrieval chain orchestration
- **ChromaDB** — vector store with metadata filtering
- **Ollama** — local LLM (qwen3.5:9b) + embeddings (nomic-embed-text-v2-moe)
- **LLM-based Reranker** — scores relevance of retrieved chunks

## Step 1 — Imports

In [ ]:
# !pip install langchain langchain-ollama langchain-community chromadb -q

from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain.schema import Document

print("All imports successful!")

## Step 2 — Sample Policy Documents

We create realistic HR, IT, and Travel policy text. In production you'd
load these from PDFs or a document management system.

In [ ]:
policies = {
    "HR Policy": {
        "source": "HR_Policy_2025.pdf",
        "department": "HR",
        "content": """HR Policy Manual — Effective January 2025

Section 1: Leave Policy
All full-time employees are entitled to 20 days of paid time off (PTO) per year.
PTO accrues at 1.67 days per month. Unused PTO may be carried over up to a
maximum of 5 days into the next calendar year. PTO beyond the carryover limit
will be forfeited on December 31.

Sick leave is provided at 10 days per year and does not accrue or carry over.
A doctor's note is required for absences exceeding 3 consecutive business days.

Section 2: Remote Work
Employees may work remotely up to 3 days per week with manager approval.
Remote work requests must be submitted through the HR Portal at least 48 hours
in advance. Fully remote arrangements require VP-level approval and a signed
Remote Work Agreement. All remote employees must be available during core hours
(10 AM – 4 PM local time) and maintain a stable internet connection.

Section 3: Performance Reviews
Performance reviews are conducted bi-annually in June and December.
Employees are rated on a 5-point scale: Exceptional (5), Exceeds Expectations (4),
Meets Expectations (3), Needs Improvement (2), Unsatisfactory (1).
Employees rated 2 or below are placed on a 90-day Performance Improvement Plan (PIP).
Promotion eligibility requires at least two consecutive ratings of 4 or above.

Section 4: Onboarding
New employees must complete mandatory onboarding within the first 5 business days.
Onboarding includes: IT setup, security training, compliance modules, team
introductions, and benefits enrollment. Buddy assignment is mandatory for all
new hires during their first 90 days."""
    },
    "IT Security Policy": {
        "source": "IT_Security_Policy_v3.pdf",
        "department": "IT",
        "content": """IT Security Policy — Version 3.2

Section 1: Password Requirements
All passwords must be at least 14 characters and include uppercase, lowercase,
numbers, and special characters. Passwords must be changed every 90 days.
Previous 12 passwords may not be reused. Multi-factor authentication (MFA) is
required for all systems classified as Tier 1 or Tier 2. MFA bypass is not
permitted under any circumstances.

Section 2: Device Management
All company-issued devices must have endpoint protection software installed.
Personal devices used for work (BYOD) must be registered with IT and comply
with the Mobile Device Management (MDM) policy. USB storage devices are
prohibited on company networks. Lost or stolen devices must be reported to
IT within 2 hours of discovery.

Section 3: Data Classification
Data is classified into four levels: Public, Internal, Confidential, and
Restricted. Confidential and Restricted data must be encrypted at rest and
in transit. Access to Restricted data requires approval from the Data Owner
and the CISO. Sharing Restricted data externally requires a signed NDA and
DLP review.

Section 4: Incident Response
Security incidents must be reported to security@company.com within 1 hour.
The incident response team will triage within 4 hours and provide initial
assessment within 24 hours. Critical incidents (data breach, ransomware)
trigger the Crisis Management Protocol and require board notification within
48 hours."""
    },
    "Travel Policy": {
        "source": "Travel_Expense_Policy.pdf",
        "department": "Finance",
        "content": """Travel & Expense Policy — Updated March 2025

Section 1: Travel Approval
All business travel must be pre-approved through the Travel Portal. Domestic
travel requires manager approval. International travel requires VP approval
and must be submitted at least 14 business days in advance. Travel to
high-risk countries requires additional approval from the Security team.

Section 2: Flight Booking
Economy class is standard for all domestic flights and international flights
under 6 hours. Business class is permitted for international flights over
6 hours with VP approval. First class is not reimbursable. Flights must be
booked through the approved travel agency (TravelCo) at least 7 days in
advance. Loyalty program preferences are permitted as long as the fare does
not exceed the lowest available fare by more than 10%.

Section 3: Hotel Policy
Hotel bookings must not exceed $200/night for domestic travel and $300/night
for international travel. Exceptions for high-cost cities (NYC, SF, London,
Tokyo) allow up to $350/night. Hotels must be booked through TravelCo or
approved platforms.

Section 4: Meal & Expense Limits
Daily meal allowance is $75 domestic and $100 international. Alcohol is not
reimbursable unless part of a client entertainment event (capped at $50/person).
All expenses over $25 require itemized receipts. Expense reports must be
submitted within 10 business days of trip completion."""
    }
}

print(f"Loaded {len(policies)} policy documents:")
for name, data in policies.items():
    words = len(data['content'].split())
    print(f"  📄 {name} ({data['department']}) — {words} words")

## Step 3 — Chunk and Index with Metadata

Each chunk gets metadata: **policy name**, **department**, **source file**.
This lets us cite exactly where an answer came from.

In [ ]:
# Create documents with rich metadata
all_documents = []
for policy_name, data in policies.items():
    doc = Document(
        page_content=data["content"],
        metadata={
            "policy": policy_name,
            "department": data["department"],
            "source": data["source"],
        },
    )
    all_documents.append(doc)

# Split into chunks while preserving section boundaries
splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=100,
    separators=["\nSection", "\n\n", "\n", ". "],
)

chunks = splitter.split_documents(all_documents)
print(f"Split {len(all_documents)} policies into {len(chunks)} chunks")

# Show chunk metadata
for i, chunk in enumerate(chunks[:3]):
    print(f"\nChunk {i}: [{chunk.metadata['policy']}]")
    print(f"  {chunk.page_content[:100]}...")

In [ ]:
# Build vector store
embeddings = OllamaEmbeddings(model="nomic-embed-text-v2-moe")

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="policy_assistant",
)

# Initial retriever fetches top-10 (we'll rerank down to top-4)
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

print(f"Vector store ready: {vectorstore._collection.count()} vectors")

## Step 4 — Build the LLM-Based Reranker

### Why Rerank?

Embedding similarity is fast but imprecise. A question like
"Can I fly business class?" might retrieve chunks about "flight booking"
AND "business continuity" — the word "business" matches both.

A reranker evaluates each (question, chunk) pair more carefully:

```
Step 1: Embeddings retrieve 10 chunks (fast, recall-oriented)
Step 2: Reranker scores all 10 and keeps the best 4 (slow, precision-oriented)
```

We use the LLM itself as a reranker — it reads the query and chunk and
assigns a relevance score from 0-10.

In [ ]:
llm = ChatOllama(model="qwen3.5:9b", temperature=0.0)

rerank_prompt = ChatPromptTemplate.from_template(
    """Rate how relevant the following document chunk is to the given question.
Return ONLY a single integer from 0 to 10, where:
- 0 = completely irrelevant
- 5 = somewhat related but doesn't answer the question
- 10 = directly answers or is critical to answering the question

Question: {question}

Document chunk:
{chunk}

Relevance score (0-10):"""
)

rerank_chain = rerank_prompt | llm | StrOutputParser()


def rerank_documents(
    question: str,
    documents: list[Document],
    top_k: int = 4,
) -> list[Document]:
    """Rerank documents by relevance to the question using LLM scoring."""
    scored = []
    for doc in documents:
        raw_score = rerank_chain.invoke({
            "question": question,
            "chunk": doc.page_content[:500],
        })
        # Extract the numeric score
        try:
            score = int("".join(c for c in raw_score.strip() if c.isdigit())[:2])
            score = min(score, 10)
        except (ValueError, IndexError):
            score = 5
        scored.append((score, doc))
    
    # Sort by score descending, take top-k
    scored.sort(key=lambda x: x[0], reverse=True)
    return [doc for _, doc in scored[:top_k]]


print("Reranker ready!")

## Step 5 — Test Retrieval vs Retrieval + Reranking

In [ ]:
question = "Can I fly business class on international flights?"

# Step 1: Raw retrieval
raw_docs = retriever.invoke(question)
print(f"🔍 RAW RETRIEVAL (top 10):")
for i, doc in enumerate(raw_docs):
    print(f"  {i+1}. [{doc.metadata['policy']}] {doc.page_content[:80]}...")

# Step 2: Rerank
print(f"\n⚡ AFTER RERANKING (top 4):")
reranked_docs = rerank_documents(question, raw_docs, top_k=4)
for i, doc in enumerate(reranked_docs):
    print(f"  {i+1}. [{doc.metadata['policy']}] {doc.page_content[:80]}...")

## Step 6 — Build the Policy Q&A Chain with Citations

In [ ]:
qa_prompt = ChatPromptTemplate.from_template(
    """You are a corporate policy assistant. Answer the employee's question based ONLY
on the policy documents provided below.

RULES:
- Answer clearly and specifically, citing exact numbers/limits from the policies
- After your answer, show CITATIONS listing which policy and section you referenced
- If multiple policies are relevant, mention all of them
- If no policy covers the question, say so clearly
- Do NOT make up policies or rules that aren't in the provided context

Format your citations as:
📎 Source: [Policy Name] — [Source File] — Section: [Section Name]

Policy excerpts:
{context}

Employee question: {question}

Answer:"""
)


def ask_policy(question: str) -> None:
    """Ask a question across all policy documents with reranking."""
    print(f"❓ {question}")
    print("-" * 60)
    
    # Retrieve
    raw_docs = retriever.invoke(question)
    
    # Rerank
    best_docs = rerank_documents(question, raw_docs, top_k=4)
    
    # Build context with metadata
    context_parts = []
    for doc in best_docs:
        meta = doc.metadata
        context_parts.append(
            f"[From: {meta['policy']} | File: {meta['source']}]\n{doc.page_content}"
        )
    context = "\n\n---\n\n".join(context_parts)
    
    # Generate answer
    qa_chain = qa_prompt | llm | StrOutputParser()
    answer = qa_chain.invoke({"context": context, "question": question})
    
    print(f"\n{answer}")
    print("=" * 60 + "\n")


print("Policy assistant ready!")

## Step 7 — Ask Policy Questions!

In [ ]:
ask_policy("How many PTO days do I get and can I carry them over?")

In [ ]:
ask_policy("What are the password requirements and do I need MFA?")

In [ ]:
ask_policy("What's the hotel limit for a trip to New York City?")

In [ ]:
# Cross-policy question — tests ability to search across departments
ask_policy("I need to travel internationally. What approvals do I need and what security requirements apply?")

In [ ]:
# Question that should get "not found" response
ask_policy("What is the company's parental leave policy?")

## Step 8 — Department-Filtered Search

ChromaDB supports **metadata filtering** — we can restrict search to
a specific department's policies.

In [ ]:
def ask_department(question: str, department: str) -> None:
    """Ask a question filtered to a specific department's policies."""
    print(f"❓ {question}")
    print(f"📁 Department filter: {department}")
    print("-" * 60)
    
    # Filtered retriever
    filtered_retriever = vectorstore.as_retriever(
        search_kwargs={
            "k": 5,
            "filter": {"department": department},
        }
    )
    docs = filtered_retriever.invoke(question)
    
    context = "\n\n---\n\n".join(
        f"[{d.metadata['policy']}]\n{d.page_content}" for d in docs
    )
    
    qa_chain = qa_prompt | llm | StrOutputParser()
    answer = qa_chain.invoke({"context": context, "question": question})
    print(f"\n{answer}")
    print("=" * 60 + "\n")


ask_department("How often should I change my password?", department="IT")

## 🧠 Key Concepts Recap

| Concept | What it does |
|---------|-------------|
| **Multi-document indexing** | All policies in one vector store with metadata |
| **Metadata tagging** | Each chunk tagged with policy name, department, source file |
| **LLM-based reranking** | Second pass scoring for precision over raw similarity |
| **Retrieve → Rerank → Generate** | Three-stage pipeline for better answers |
| **Metadata filtering** | Restrict search to specific departments |
| **Structured citations** | Show exactly which policy section was referenced |

## 🔧 Customization Ideas

- **Cross-encoder reranker**: Use `sentence-transformers` CrossEncoder for faster reranking
- **Policy versioning**: Track multiple versions of the same policy
- **Conflict detection**: Flag contradictory rules across policies
- **Slack/Teams integration**: Deploy as a chatbot employees can query